# 2026/10/20 VJU Intern Lesson1

### We use observational data from the Himawari meteorological satellite to visualize clouds.

In [ ]:
# Import the libraries to be used
import numpy as np
import pandas as pd
import subprocess
import os
import math
import bz2
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from datetime import datetime, timedelta


In [ ]:
# Common settings
shape = (3000, 3000)
west_japan = (slice(550, 750), slice(1000, 1400))
kochi_area = (slice(625, 675), slice(1150, 1250))

# Date
date = "20260902"
time = "0000" #UTC Time

# Himawari data download info.
url = f"ftp://hmwr829gr.cr.chiba-u.ac.jp/gridded/FD/V20190123/202609/4KM/{date}/"
band_list = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B13']
name_list = ['vis.01', 'vis.02', 'ext.01', 'vis.03', 'sir.01', 'sir.02', 'tir.01']
wave_list = ['0.47μm', '0.51μm', '0.64μm', '0.86μm', '1.6μm', '2.3μm', '10.4μm']

# Cloud height estimation.
h = 6.626e-34           # Planck constant [J s]
c = 3.0e8               # Speed ​​of light [m/s]
k = 1.381e-23           # Boltzmann constant [J/K]
lambda_m = 10.4e-6      # wavelength [m]
T_surface = 305.0       # Surface temperature [K]
lapse_rate = 6.5        # Temperature lapse rate [K/km]


# Cloud thickness estimation.
rho_w = 1000            # Density of water [kg/m³]
r_eff = 10e-6           # Effective radius of cloud droplets [m]
LWC = 0.0005            # Liquid water content [kg/m³]

# Data loading function.
def load_bz2(fname, shape=shape):
    with bz2.BZ2File(fname) as f:
        image = np.frombuffer(f.read(), '>f4').reshape(shape)
        image = image.astype(np.float32)
        return image

# Contrast correction and gamma correction functions.
def correct_image(image):
    vmin = np.percentile(image , 10)
    vmax = np.percentile(image , 90)
    image  = np.clip((image - vmin) / (vmax - vmin + 1e-6), 0, 1)
    image = image ** 0.5
    return(image)

# Conversion from radiance (captured light intensity) to brightness temperature.
def Brightness_Temperature(image):
    image = image * 1e6
    term1 = (2 * h * c**2) / (lambda_m**5)
    term2 = term1 / image
    T = (h * c) / (lambda_m * k * np.log(term2 + 1))
    return T

# Convert brightness temperature to altitude (the lapse rate decreases with every 1 km increase in altitude).
def Cloud_Top(T):
    return (T_surface - T) / lapse_rate

# Convert brightness temperature to optical depth (tau) and physical thickness (thick).
def Thick(T):
    tau = 0.5 * T + 1
    thick = (2 * rho_w * r_eff * tau) / (3 * LWC) /1000
    return thick


In [ ]:
#File download
file_list = {}

for i in range(len(band_list)):
    file_list[i] = f"{date}{time}.{name_list[i]}.rad.fld.4km.bin.bz2"
    subprocess.run(["curl", "-O", url + file_list[i]], check=True)

In [ ]:
# Show Data
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes_flat = axes.ravel()

for i in range(6):
    ax = axes_flat[i]
    image = correct_image(load_bz2(file_list[i]))
    ax.imshow(image, cmap='gray')
    ax.set_title(band_list[i])
    ax.axis('off')

plt.show()


In [ ]:
# Combine bands 1, 2, and 3
r = correct_image(load_bz2(file_list[band_list.index('B03')]))
g = correct_image(load_bz2(file_list[band_list.index('B02')]))
b = correct_image(load_bz2(file_list[band_list.index('B01')]))
image = np.stack([r, g, b], axis=-1)

plt.figure(figsize=(12, 12))
plt.imshow(image)
plt.show()


In [ ]:
# Visualize moisture distribution
r = correct_image(load_bz2(file_list[band_list.index('B05')]))
g = correct_image(load_bz2(file_list[band_list.index('B04')]))
b = correct_image(load_bz2(file_list[band_list.index('B03')]))
image = np.stack([r, g, b], axis=-1)

plt.figure(figsize=(12, 12))
plt.imshow(image)
plt.show()

In [ ]:
# Display only the specified area
plt.imshow(image[west_japan])
plt.show()
plt.imshow(image[kochi_area])
plt.show()

kochi = image[kochi_area][:,:,1]


# [Q] How can I obtain solar irradiance data for Kochi from these satellite images?

In [ ]:
# Cloud detection
threshold=0.35

image_cloud = correct_image(load_bz2(file_list[band_list.index('B03')]))
image_cloud[image_cloud>threshold] = 1
image_cloud[image_cloud<=threshold] = 0

plt.imshow(image_cloud[kochi_area], cmap='gray')
plt.show()

In [ ]:
# Calculate cloud height

image = load_bz2(file_list[band_list.index('B13')])
image = image[west_japan][::4, ::4]
image = cv2.GaussianBlur(image, (5, 5), 1)
height = Cloud_Top(Brightness_Temperature(image))

X = np.arange(height.shape[1])
Y = np.arange(height.shape[0])
Z = height

fig = go.Figure()

# Cloud-height surface
fig.add_trace(
    go.Surface(
        x=X, y=Y, z=np.flipud(Z),
        colorscale="jet", opacity=0.6
    )
)

# Bottom-surface map
fig.add_trace(
    go.Surface(
      x=X, y=Y, z=np.full_like(Z, -10),
      surfacecolor=np.flipud(kochi), colorscale="Gray", opacity=0.6,
      showscale=False
    )
)

fig.update_layout(
    scene = {
        "aspectratio": {"x": 2, "y": 1, "z": 1},
        "camera": {"eye": {"x": 0, "y": -1, "z": 1}}
    }
)
fig.show()


In [ ]:
# Calculate cloud thickness
image = load_bz2(file_list[band_list.index('B06')])
image = image[west_japan][::4, ::4]
image = cv2.GaussianBlur(image, (5, 5), 10)
thickness = Thick(Brightness_Temperature(image))

# Extract the top and bottom edges only where clouds are present
X = np.arange(height.shape[1])
Y = np.arange(height.shape[0])
X, Y = np.meshgrid(X, Y)
Z = thickness

base = height - thickness
mask = (height > 2) & (thickness > 0.5)

Z1 = np.where(mask, height, np.nan)
Z2 = np.where(mask, base, np.nan)

z1 = np.flipud(Z1).ravel()
z2 = np.flipud(Z2).ravel()
x_line = np.repeat(X.ravel(), 3)
y_line = np.repeat(Y.ravel(), 3)
z_line = np.c_[z1, z2, np.full_like(z1, None)].ravel()

fig = go.Figure()

# Drawing clouds
fig.add_trace(
    go.Scatter3d(
        x=x_line, y=y_line, z=z_line,
        mode='lines', line=dict(color='rgba(0,0,100,0.2)', width=10),
        showlegend=False
    )
)

# Bottom-surface map
fig.add_trace(
    go.Surface(
      x=X, y=Y, z=np.full_like(Z, -10),
      surfacecolor=np.flipud(kochi), colorscale="Gray", opacity=0.6,
      showscale=False
    )
)

fig.update_layout(
    scene = {
        "aspectratio": {"x": 2, "y": 1, "z": 1},
        "camera": {"eye": {"x": 0, "y": -1, "z": 1}}
    }
)
fig.show()

In [ ]:
# Cloud distribution as viewed from the ground

x_size, y_size = 1200, 600
x0, y0, z0 = 50, 25, 0
dx = (X - x0) * (x_size / (height.shape[1] - 1))
dy = (Y - y0) * (y_size / (height.shape[0] - 1))
u, v = dx / (z0 - Z1 + 1e-6), dy / (z0 - Z2 + 1e-6)
u2, v2 = dx / (z0 - Z2 + 1e-6), dy / (z0 - Z2 + 1e-6)

fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(u, v, color='blue', s=10)
ax.scatter(u2, v2, color='skyblue', s=5)

for angle in [1, 2, 5]:
    r = np.cos(np.radians(angle))/np.sin(np.radians(angle))
    ax.add_patch(plt.Circle((0, 0), r, fill=False))

fig.text(0.5, 0.12, 'South 1°', ha='center', va='bottom', fontsize=12, color="red")
ax.set_xlim(-80, 80)
ax.set_ylim(-80, 80)
plt.show()